# Colorado reservoir storage — liberation pipeline

One notebook, four stages: **Retrieve → Audit → Cleanup → Publish-to-CSV**, thin
orchestration over the tested `reservoir` package (logic lives in `src/reservoir/`,
so it is unit-tested and reusable). Implements issue
[#9](https://github.com/CUPIDS-Lab/co-environmental-data/issues/9).

| Stage | Does | Writes |
|---|---|---|
| 1 · Retrieve | pull raw API responses (immutable) | `data/original/` + manifest |
| 2 · Audit | profile the retrieval | `data/audit/raw-profile-*.md` |
| 3 · Cleanup | parse → tidy long → validate | `data/processed/reservoir-storage.csv` + provenance |
| 4 · Publish | finalize CSV + audit + reconcile | `data/audit/summary-*.md`, `docs/variables.csv` |

**Run order:** top to bottom. `uv sync` first (or `pip install -e .`).

> **CDSS notes (confirmed live, 2026-06):** the value field is `measValue`; reservoir
> `abbrev`s are codes (Green Mountain → `GRERESCO`); CDSS returns **HTTP 404 for any
> zero-record query** — `fetch` treats that as *no data*, not an error, so one empty
> series never crashes the run.

In [ ]:
# Make the `reservoir` package importable (cleaner: `uv pip install -e ..`).
import sys, pathlib
SRC = pathlib.Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from reservoir import config, fetch, clean, audit
from IPython.display import Markdown
import pandas as pd, json
registry = config.get_sources(); list(registry)

## 1 · Retrieve  *(RETRIEVAL)*

`discover()` is offline — it builds the API request URLs without downloading.
Inspect the plan, then fetch. Originals land in `data/original/` and are immutable.

In [ ]:
plan = [{'source': s, 'reservoir': a.metadata.get('reservoir_name'), 'file': a.local_path.name}
        for s, src in registry.items() for a in src.discover()]
plan_df = pd.DataFrame(plan)
print(f'{len(plan_df)} artifacts planned'); plan_df.groupby('source').size()

**Live vs. offline.** `LIVE = True` pulls from the real APIs. **DWR/CDSS works now**
(confirmed endpoint + abbrevs); **RISE** needs catalog item ids in `reservoirs.csv`
and **Northern Water** needs its FeatureServer URL in `sources.yaml` (both marked
`⚠️ VERIFY`, skipped safely until filled). `LIVE = False` seeds the bundled fixture
so stages 2–4 run with no network.

In [ ]:
LIVE = False  # ← True for a real pull (DWR returns data today; RISE/Northern skip until VERIFY filled)

if LIVE:
    fetched = fetch.fetch_all()      # 404 = zero-records is handled as no-data, never fatal
else:
    import shutil
    fx = config.PROJECT_DIR / 'tests' / 'fixtures' / 'dwr_cdss_storage_sample.json'
    art = next(a for a in registry['dwr_cdss'].discover()       # the Green Mountain storage artifact
               if a.metadata.get('reservoir_id') == 'GRERESCO' and a.local_path.name.endswith('STORAGE.json'))
    art.local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(fx, art.local_path)
    print('offline demo seeded:', art.local_path.relative_to(config.PROJECT_DIR))

In [ ]:
files = sorted(p for p in config.ORIGINAL.rglob('*.json') if p.name != 'manifest.json')
print(f'{len(files)} raw files in data/original/')
assert files, 'No raw files — run the fetch cell (LIVE or offline).'
if (config.AUDIT/'fetch_errors.json').exists():
    print('fetch errors:', len(json.loads((config.AUDIT/'fetch_errors.json').read_text())))

## 2 · Audit  *(profile the retrieval)*

Before cleaning, profile what came back: which sources produced data, how many
files, what the response shapes are. Catches an empty/failed pull early.

In [ ]:
Markdown(audit.profile_raw())

## 3 · Cleanup  *(TIDY)*

Route each response through its parser → **tidy long** (one row per
`source · reservoir_id · datetime · variable`), harmonize units, preserve QA flags,
validate against the pandera schema, write the CSV + provenance sidecar. Errors are
durable (a failed parse logs to `extraction_errors.json` and the run continues).

In [ ]:
data, prov = clean.run(fail_on_empty=False)
print('rows:', len(data), '| reservoirs:', data['reservoir_id'].nunique(),
      '| sources:', data['source'].nunique())
data.head(8)

In [ ]:
prov

In [ ]:
errs = config.AUDIT / 'extraction_errors.json'
print(errs.read_text() if errs.exists() else 'no extraction errors ✓')

## 4 · Publish to CSV  *(PUBLISH)*

The tidy CSV is the deliverable. Finalize: full processed audit, the auto-generated
column profile (the data dictionary's sanity check), and reconciliation against each
agency's published totals. (Datasette/Quarto surfaces are deferred to the Hub roadmap.)

In [ ]:
Markdown(audit.audit_processed())

In [ ]:
audit.variables_report()

Fill `expected` with current storage off each agency's page (see `docs/survey-notes.md`); any mismatch beyond tolerance is a regression.

In [ ]:
expected = {
    # ('dwr_cdss', 'GRERESCO'): 57176.7,   # ← confirm from the DWR station page
}
audit.reconcile(expected) if expected else print('reconcile: fill expected totals (survey-notes)')

In [ ]:
csv = config.CANONICAL_CSV
print('deliverable:', csv.relative_to(config.PROJECT_DIR))
print('load: pd.read_csv(%r, parse_dates=[\'datetime\'])' % csv.name)
print('wide views -> docs/filter-pivot-recipes.md')